In [3]:
##CURATING BRAND SPECIFIC DATA

import pandas as pd
import pickle
import re

# Added ../ here so it finds your file
df = pd.read_csv('../data/processed/clean_data.csv')

# Define brands to analyze
brands = {
    'apple':    ['apple', 'iphone', 'macbook', 'ipad', 'airpods'],
    'google':   ['google', 'gmail', 'android', 'youtube', 'goog'],
}

brand_dfs = {}
for brand, keywords in brands.items():
    pattern = '|'.join(keywords)
    mask = df['text'].str.lower().str.contains(pattern, na=False)
    brand_dfs[brand] = df[mask].copy()
    print(f'{brand}: {len(brand_dfs[brand])} tweets')

apple: 784 tweets
google: 447 tweets


In [4]:
##APPLY YOUR MODEL TO EACH BRAND

# Load saved model and vectorizer
with open('../models/tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)
with open('../models/sentiment_model.pkl', 'rb') as f:
    model = pickle.load(f)

for brand, bdf in brand_dfs.items():
    vec = tfidf.transform(bdf['clean_text'])
    bdf['predicted_sentiment'] = model.predict(vec)
    bdf['confidence'] = model.predict_proba(vec).max(axis=1)
    brand_dfs[brand] = bdf
    print(f'{brand} sentiment breakdown:')
    print(bdf['predicted_sentiment'].value_counts())
    print()

apple sentiment breakdown:
predicted_sentiment
0    540
1    244
Name: count, dtype: int64

google sentiment breakdown:
predicted_sentiment
1    290
0    157
Name: count, dtype: int64



In [5]:
##TOPIC MODELING WITH LDA

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# Define the helper function to extract topics
def get_topics(texts, n_topics=5, n_words=8):
    """Extract top topics from a list of texts"""
    cv = CountVectorizer(max_features=5000, min_df=2, stop_words='english')
    dtm = cv.fit_transform(texts)
    
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        max_iter=10
    )
    lda.fit(dtm)
    
    words = cv.get_feature_names_out()
    topics = []
    for topic_idx, topic in enumerate(lda.components_):
        top_words = [words[i] for i in topic.argsort()[:-n_words-1:-1]]
        topics.append(top_words)
    return topics

# Run for each brand and sentiment group
for brand, bdf in brand_dfs.items():
    print(f'\n=== {brand.upper()} ===')
    
    # Check what sentiment labels your model uses (Sentiment140 usually uses 0 and 4)
    # If your previous output showed 1 instead of 4 for positive, change 4 below to 1.
    for sentiment, label in [(4, 'POSITIVE'), (0, 'NEGATIVE')]:
        subset = bdf[bdf['predicted_sentiment'] == sentiment]['clean_text']
        
        # Skip if there aren't enough tweets to extract meaningful topics
        if len(subset) < 50:
            continue
            
        topics = get_topics(subset.tolist())
        print(f'\n  {label} topics:')
        for i, t in enumerate(topics, 1):
            print(f'    Topic {i}: {", ".join(t)}')



=== APPLE ===

  NEGATIVE topics:
    Topic 1: iphone, apple, got, doesnt, like, app, going, tweetdeck
    Topic 2: iphone, new, apple, os, oh, store, im, tomorrow
    Topic 3: iphone, apple, iphones, wont, dont, macbook, working, wants
    Topic 4: iphone, im, new, apple, macbook, dont, time, update
    Topic 5: iphone, squarespace, gs, new, wish, like, friday, feel

=== GOOGLE ===

  NEGATIVE topics:
    Topic 1: youtube, trying, dont, video, wont, let, lol, believe
    Topic 2: gmail, youtube, need, really, missed, hate, im, miss
    Topic 3: youtube, im, watch, videos, watching, work, account, want
    Topic 4: youtube, google, doesnt, clip, like, know, wish, sad
    Topic 5: google, phone, site, im, chrome, search, dont, wanna


In [6]:
##TIME-SERIES SENTIMENT TREND

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

for brand, bdf in brand_dfs.items():
    # Parse dates
    bdf['date'] = pd.to_datetime(
        bdf['date'],
        format='%a %b %d %H:%M:%S PDT %Y',
        errors='coerce'
    )
    bdf = bdf.dropna(subset=['date'])
    bdf['week'] = bdf['date'].dt.to_period('W')
    
    # Compute weekly sentiment ratio
    weekly = bdf.groupby('week').agg(
        total=('predicted_sentiment', 'count'),
        positive=('predicted_sentiment', lambda x: (x == 4).sum()) # Adjusted to count Sentiment140 positive label (4)
    ).reset_index()
    
    weekly['positive_pct'] = (weekly['positive'] / weekly['total']) * 100
    weekly['week_dt'] = weekly['week'].dt.to_timestamp()
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.fill_between(
        weekly['week_dt'], 
        weekly['positive_pct'], 
        alpha=0.3, 
        color='#2563EB'
    )
    ax.plot(
        weekly['week_dt'], 
        weekly['positive_pct'], 
        color='#2563EB', 
        linewidth=2
    )
    
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    ax.set_title(f'{brand.capitalize()} - Weekly Sentiment %', fontsize=14)
    ax.set_ylabel('% Positive Tweets')
    ax.set_ylim(0, 100)
    plt.tight_layout()
    
    # Saves the graph automatically to your reports folder
    plt.savefig(f'../reports/07_{brand}_sentiment_trend.png', dpi=150, bbox_inches='tight')
    plt.show()


KeyError: 'date'

In [7]:
print(df.columns)

Index(['text', 'sentiment', 'clean_text'], dtype='str')


In [8]:
# 5.5 Generate a Brand Insight Summary
for brand, bdf in brand_dfs.items():
    total = len(bdf)
    
    # Adjusting to count your specific labels: 4 = Positive, 0 = Negative
    pos = (bdf['predicted_sentiment'] == 4).sum()
    neg = (bdf['predicted_sentiment'] == 0).sum()
    avg_conf = bdf['confidence'].mean()
    
    # Prints out the perfect executive report format required by the guide
    print('\n' + '='*40)
    print(f'  BRAND REPORT: {brand.upper()}')
    print('='*40)
    print(f'Total mentions analyzed: {total}')
    print(f'Positive: {pos} ({pos/total*100:.1f}%)' if total > 0 else 'Positive: 0')
    print(f'Negative: {neg} ({neg/total*100:.1f}%)' if total > 0 else 'Negative: 0')
    print(f'Avg model confidence: {avg_conf:.2%}')
    print()
    
    # Save the brand-specific data files (../ steps out of the notebooks folder)
    bdf.to_csv(f'../data/processed/{brand}_analyzed.csv', index=False)


  BRAND REPORT: APPLE
Total mentions analyzed: 784
Positive: 0 (0.0%)
Negative: 540 (68.9%)
Avg model confidence: 74.07%


  BRAND REPORT: GOOGLE
Total mentions analyzed: 447
Positive: 0 (0.0%)
Negative: 157 (35.1%)
Avg model confidence: 74.03%

